Independently of their specific type (interest rates, volatility, inflation or whatnot) the term structures implemented in the library share the management of their reference date; in particular, they can be set up so that they track (or don’t track) the global evaluation date. This notebook shows an example of this.

In [1]:
import QuantLib as ql

In [2]:
ql.Settings.instance().evaluationDate=ql.Date(3, ql.October,2024)

<h2>Specifying the reference date of a term structure</h2>

I’ll do it here by building a simple flat interest-rate curve: note the 0 and TARGET() arguments, specifying the number of days and the calendar used to determine business days.

In [3]:
r=ql.SimpleQuote(0.05)

In [4]:
curve1 = ql.FlatForward(0, ql.TARGET(), ql.QuoteHandle(r), ql.Actual360())

The second possibility is to specify the reference date explicitly; in the cell below I use a second overload of the FlatForward class takes a given date instead of the reference days and calendar. 

In [12]:
curve2 = ql.FlatForward(
    ql.Date(3, ql.October, 2024),ql.QuoteHandle(r), ql.Actual360()
)

In [6]:
print(curve1.referenceDate())

October 3rd, 2024


In [13]:
print(curve2.referenceDate())

October 3rd, 2024


return the discount factors, whether we ask for a given time (for instance,t =5

In [14]:
print(curve1.discount(5.0))
print(curve2.discount(5.0))

0.7788007830714049
0.7788007830714049


…or for a given date: for instance, 5 years from the reference date, which will be different from the above because the corresponding time is greater than 5 for the Actual/360 day count convention we passed to the curve.

In [15]:
print(curve1.discount(ql.Date(3, ql.October, 2029)))
print(curve2.discount(ql.Date(3, ql.October, 2029)))

0.7759935186328056
0.7759935186328056


<h2>Moving the evaluation date</h2>

In [17]:
ql.Settings.instance().evaluationDate=ql.Date(19, ql.September, 2024)

In [18]:
print(curve1.referenceDate())
print(curve2.referenceDate())

September 19th, 2024
October 3rd, 2024


And of course the discount factors have changed, too…

In [19]:
print(curve1.discount(5.0))
print(curve2.discount(5.0))

0.7788007830714049
0.7788007830714049


…but only if we look at them in the right way. The whole curve has moved back a couple of weeks, so if we ask for a given time (that is, a given time from the reference) we’ll get the same discount factor; in other words, we’re asking for the discount factor over five years after the reference date, and that remains the same for a rigid translation of the curve. If we ask for the discount factor at a given date, though, we’ll see the effect:

In [20]:
print(curve1.discount(ql.Date(3, ql.October, 2029)))
print(curve2.discount(ql.Date(3, ql.October, 2029)))

0.7744861083592839
0.7759935186328056


<h2>Notifications</h2>

Finally, we can see that the two curves behave differently also with respect to notifications. Let’s make two observers: each of them will output a message when they receive a notification.

In [24]:
def make_observer(i):
    def say():
        s = "Observer %d notified" % i
        print("-"*len(s))
        print(s)
        print("-"*len(s))

    return ql.Observer(say)

obs1 = make_observer(1)
obs2 = make_observer(2)

And now, let’s connect each of them to one of the curves. The first observer will receive notifications from the curve that moves with the evaluation date, and the second observer will receive notifications from the curve that doesn’t move.

In [25]:
obs1.registerWith(curve1)
obs2.registerWith(curve2)

In [26]:
r.setValue(0.03)

-------------------
Observer 1 notified
-------------------
-------------------
Observer 1 notified
-------------------
-------------------
Observer 2 notified
-------------------
-------------------
Observer 2 notified
-------------------


Instead, we can see what happens when the evaluation date changes again:

In [27]:
ql.Settings.instance().evaluationDate=ql.Date(23, ql.September, 2024)

-------------------
Observer 1 notified
-------------------


As you can see, only the moving curve sent a notification. The other did not, since it was not modified by the change of evaluation date.